In [ ]:
# [auto] project-root setup
import os, sys
from pathlib import Path

# Walk upward to find the project root (folder containing .gitignore)
_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / '.gitignore').exists():
    _p = _p.parent
PROJECT_ROOT = _p

# Switch cwd to the project root so every relative path (Stage1_Exploration/, Refined_Results_v4/, ...) keeps working
os.chdir(PROJECT_ROOT)
# Let the notebooks do `from viz_config import VizConfig`
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
print(f'[setup] PROJECT_ROOT = {PROJECT_ROOT}')


In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# 0. Dataset construction script
# ==========================================
# Purpose: merge the raw CFD results (summary table) with the case parameters (cases table) 
#       and clean them into a ML-ready dataset (train_dataset_ready.csv).
# Inputs: 
#   - cases.csv: Per-case inlet conditions (V_in, C_in, Area)
#   - summary_0_499.csv: Per-case concentration profile along the domain
# Outputs:
#   - train_dataset_ready.csv: Merged long-format dataset

# 1. Load the raw data
print("Reading raw data files...")
try:
    df_cases = pd.read_csv('data/cases.csv')
    df_summary = pd.read_csv('data/summary_0_499.csv')
except FileNotFoundError as e:
    print(f"ERROR: file {e.filename} not found. Make sure cases.csv and summary_0_499.csv sit in the current directory.")
    exit()

# ==========================================
# 2. Data reshaping (wide -> long)
# ==========================================
# The raw summary table is in wide format (one column per distance point), which is not ideal for training.
# Use melt() to convert it to long format (one sample per row).

print("Running the melt step...")
df_long = df_summary.melt(id_vars=['Case'], 
                          var_name='Distance', 
                          value_name='C_out')

# 3. Type casting
# Distance Column names are strings by default - cast them to floats for the downstream math.
df_long['Distance'] = df_long['Distance'].astype(float)

# ==========================================
# 4. Feature merging
# ==========================================
# Join the per-case parameters (V_in, C_in, Area) onto every sample via Case ID.
# Each sample then carries both its full input features and its target output.

print("Merging case parameters...")
df_merged = pd.merge(df_long, df_cases, on='Case', how='left')

# ==========================================
# 5. Cleaning and export
# ==========================================
feature_cols = ['V_in', 'C_in', 'Area', 'Distance']
target_col = ['C_out']

# Sanity-check completeness
print(f"Merged shape:           {df_merged.shape}")
df_merged = df_merged.dropna() 
print(f"Cleaned shape (no NaN): {df_merged.shape}")

# Preview
print("\nData preview (first 5 rows):")
print(df_merged[feature_cols + target_col].head())

# Save the final dataset
output_file = 'data/train_dataset_ready.csv'
df_merged.to_csv(output_file, index=False)
print(f"\nDataset built successfully and saved to {output_file}")